# 5 自定义量化 A8W8 matmul 算子开发

本章及第6章是整个启航营的**核心实践环节**，将学习到的所有知识沉淀转化为一次完整的自定义算子开发实战。

### 本章定位

本章将开发一个自定义的量化 matmul 算子 `QmmCustom`，通过 A8W8 量化（Activation INT8 + Weight INT8）减少计算量和内存占用，从而提升推理性能。这是本次启航营中综合性最强的一章——涵盖了从算子概念理解、工程创建、Tiling 设计、Kernel 实现到编译部署、C++ 扩展接入与功能性能测试的全链路。

### 通过实践达成的学习目标

- **理解量化推理原理**：掌握 A8W8 量化方案（INT8 激活 + INT8 权重）的数学原理与 Scale 反量化机制，理解 perChannelScale 与 perTokenScale 的作用。
- **掌握算子工程创建流程**：通过 `msopgen` 工具从算子原型 JSON 自动生成算子工程框架，理解 op_host/op_kernel 的分工。
- **设计 Tiling 数据结构与函数**：根据算子原型定义自行设计 TilingData 结构体，实现多核分块的 Tiling 函数，理解 Cube 计算单元的分块策略。
- **实现 Cube+Vector 双路径 Kernel**：完成 Cube-only（INT32 输出）和 Cube+Vector 协作（反量化 BF16 输出）两条 Kernel 路径的实现，理解昇腾 AI Core 的 Cube 与 Vector 协作编程模型。
- **完成算子编译部署与 C++ 扩展接入**：将自定义算子编译打包、安装到 CANN 环境，并通过 PyTorch C++ Extension 接入框架，打通“算子开发 → 框架调用”的链路。
- **测试算子功能与性能，尝试优化迭代**：验证自定义算子的精度正确性，测试单算子性能，并根据测试结果对 Tiling 或 Kernel 进行优化迭代。

### 本章大纲

- 环境准备
- A8W8 量化 matmul 算子概念
- 算子工程创建
- Tiling 实现
    - **任务一：自行设计tilingdata结构体并实现tiling函数**
- Kernel 实现
    - **任务二：实现A8W8输入，INT32输出的kernel**
    - **任务三：实现A8W8输入，经过perChannelScale和perTokenScale反量化输出BF16的kernel**
- 编译并安装自定义算子包
    - **任务四：成功编译并安装自定义算子包**
- 基于C++扩展调用自定义算子
- 测试自定义算子功能性能
    - **任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程**
- TIPS
- 评分标准总览

---

## 1 环境准备

定位教程目录、准备运行目录，并把 CANN 环境变量导入当前 Notebook 进程：

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)

def locate_repo_root():
    repo_roots = []
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'reference_practice/model_inference_optimization/qwen3_8b/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'reference_practice' / 'model_inference_optimization' / 'qwen3_8b'
SRC_DIR = TUTORIAL_DIR / 'src' / 'op_custom'

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment.')
print('CANN包路径 =', os.environ.get('ASCEND_TOOLKIT_HOME'))
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

QMM_CUSTOM_DIR = SRC_DIR / 'qmm_custom'
QMM_EXT_DIR = SRC_DIR / 'torch_ops_extension'

print('教程目录 =', TUTORIAL_DIR)
print('相关文件目录 =', SRC_DIR)
print('算子工程目录 =', QMM_CUSTOM_DIR)
print('PyTorch扩展目录 =', QMM_EXT_DIR)
print('[环境准备] 完成', flush=True)

安装所需的依赖：

In [ ]:
%pip install -r {SRC_DIR}/requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple --trusted-host pypi.tuna.tsinghua.edu.cn

---
## 2 A8W8 量化 matmul 算子概念

### 2.1 什么是 A8W8 量化

A8W8 量化是 LLM 推理中常用的一种量化方案：
- **A（Activation）**：将输入激活值从 BF16/FP16 量化为 **INT8**
- **W（Weight）**：将权重从 BF16/FP16 量化为 **INT8**
- **反量化**：计算结果 `INT8 × INT8 → INT32` 后，通过 Scale 反量化回浮点精度

### 2.2 算子规格

通过分析量化模型的profiling，可以确定自定义算子的输入输出规格：
| 算子名称 | QmmCustom |
|------|-----|
| 输入 x1 (A) | INT8, ND 格式, shape [M, K] |
| 输入 x2 (B) | INT8, **FRACTAL_NZ** 格式, shape [K, N] |
| 输入 scale | FLOAT32, ND 格式, shape [N] |
| 输入 pertoken_scale | FLOAT32, ND 格式, shape [M]（可选） |
| 输出 y | 有 perTokenScale → BF16；无 → INT32 |
| 架构 | ascend910b |

**注意**：x2 权重矩阵使用 **FRACTAL_NZ** 格式，这是昇腾 AI Core Cube 单元的高效存储格式，能提升 matmul 计算性能。

### 2.3 Scale 模式

QmmCustom 算子支持两种 Scale：

| Scale 类型 | Shape | 作用 | 
|-----------|-------|------|
| **perChannelScale** | `[N]` | 对每个输出通道单独缩放 |
| **perTokenScale** | `[M]` | 对每行输入单独缩放 |

计算公式：
```
result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale
```

---
## 3 算子工程创建

### 3.1 算子工程创建

使用 `msopgen` 工具前，需要先编写算子原型定义文件 `QmmCustom.json`，描述算子的输入输出规格。

点击查看算子原型json文件内容：

In [ ]:
!cat {SRC_DIR}/QmmCustom.json

使用 `msopgen` 工具基于上述 json 文件自动生成算子工程框架：

In [ ]:
!rm -rf {QMM_CUSTOM_DIR}
!msopgen gen -i {SRC_DIR}/QmmCustom.json -c ai_core-ascend910b1 -lan cpp -f pytorch -out {QMM_CUSTOM_DIR}

### 3.2 工程目录结构

使用 `msopgen` 工具基于上述 json 文件自动生成算子工程框架后，目录结构如下所示：

```
qmm_custom
|--- op_host
|   |--- qmm_custom.cpp                     // 包含tiling实现、shape与dtype推导、算子原型注册
|   |--- CMakeLists.txt                     // host侧CMakeLists文件，无需修改
|--- op_kernel
|   |--- qmm_custom_tiling.h                // 算子tilingdata定义文件   
|   |--- qmm_custom.cpp                     // 算子kernel实现文件 
|   |--- CMakeLists.txt                     // kernel侧CMakeLists文件，无需修改
|--- CMakeLists.txt                         // 根目录CMakeLists文件，无需修改
|--- CMakePresets.json                      // CMake编译配置文件，无需修改
|--- build.sh                               // 编译脚本，无需修改
```

**其中需要自行修改实现的关键文件如下**：
- `op_host/qmm_custom.cpp`：包含Tiling 函数、InferShape/InferDataType、算子原型注册（OpDef）
- `op_kernel/qmm_custom_tiling.h`：TilingData 结构体定义
- `op_kernel/qmm_custom.cpp`：Kernel 侧实现，包含两条执行路径

执行以下命令查看刚刚生成的算子工程目录结构：

In [ ]:
!cd {QMM_CUSTOM_DIR}; find . -maxdepth 2 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

---
## 4 Tiling 实现

### 任务一：自行设计tilingdata结构体并实现tiling函数

### 4.1 TilingData 结构

在qmm_custom_tiling.h中定义TilingData 结构：

In [ ]:
%%writefile {QMM_CUSTOM_DIR}/op_kernel/qmm_custom_tiling.h 
#ifndef QMM_CUSTOM_TILING_H
#define QMM_CUSTOM_TILING_H

#ifndef __CCE_AICORE__
#include <cstdint>
#endif

#include "kernel_tiling/kernel_tiling.h"

#pragma pack(push, 8)
struct alignas(8) QmmCustomTilingData {
    # TODO: 自行设计tilingdata结构体，下面仅为示例
    TCubeTiling cubeTilingData;
    uint32_t isPertoken;
    ......
};
#pragma pack(pop)

#endif // QMM_CUSTOM_TILING_H

### 4.2 Tiling 函数

在op_host/qmm_custom_tiling.cpp中实现tiling函数：

In [ ]:
%%writefile {QMM_CUSTOM_DIR}/op_host/qmm_custom.cpp 
#include "../op_kernel/qmm_custom_tiling.h"
#include "register/op_def_registry.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
using namespace matmul_tiling;

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{
    # TODO: 自行实现tiling函数
    QmmCustomTilingData *tiling = context->GetTilingData<QmmCustomTilingData>();
    ......
    return ge::GRAPH_SUCCESS;
}
}


实现InferShape与InferDataType：

In [ ]:
%%writefile -a {QMM_CUSTOM_DIR}/op_host/qmm_custom.cpp
namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    int64_t M = x1_shape->GetDim(0);
    const gert::Shape* x2_shape = context->GetInputShape(1);
    // x2逻辑形状为[K, N]，N在第二维度
    int64_t N = x2_shape->GetDim(1);

    y_shape->SetDimNum(2);
    y_shape->SetDim(0, M);
    y_shape->SetDim(1, N);
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    // 根据是否有pertoken_scale决定输出类型
    auto pertokenDataType = context->GetInputDataType(3);
    if (pertokenDataType != ge::DT_UNDEFINED) {
        context->SetOutputDataType(0, ge::DT_BF16);
    } else {
        context->SetOutputDataType(0, ge::DT_INT32);
    }
    return ge::GRAPH_SUCCESS;
}
}


算子原型定义：

In [ ]:
%%writefile -a {QMM_CUSTOM_DIR}/op_host/qmm_custom.cpp 
namespace ops {
class QmmCustom : public OpDef {
public:
    explicit QmmCustom(const char* name) : OpDef(name)
    {
        this->Input("x1")
            .ParamType(REQUIRED)
            .DataType({ge::DT_INT8, ge::DT_INT8})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("x2")
            .ParamType(REQUIRED)
            .DataType({ge::DT_INT8, ge::DT_INT8})
            .Format({ge::FORMAT_FRACTAL_NZ, ge::FORMAT_FRACTAL_NZ})
            .UnknownShapeFormat({ge::FORMAT_FRACTAL_NZ, ge::FORMAT_FRACTAL_NZ});
        this->Input("scale")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("pertoken_scale")
            .ParamType(OPTIONAL)
            .DataType({ge::DT_FLOAT, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_BF16, ge::DT_INT32})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND});

        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);

        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");

    }
};

OP_ADD(QmmCustom);
}

---
## 5 Kernel 实现

### 5.1 双路径架构

Kernel 根据 `isPertoken` 标志分为两条执行路径：

| 路径 | 条件 | 计算流程 | 输出类型 |
|------|------|----------|----------|
| **Path 1: Cube-only** | `isPertoken == 0` | INT8 matmul → INT32 | INT32 |
| **Path 2: Cube+Vector** | `isPertoken == 1` | INT8 matmul → INT32 workspace → dequant(scale×pertoken) → BF16 | BF16 |

**Path 1**：纯 Cube 计算，无 Vector 参与，直接输出 INT32 结果。

**Path 2**：Cube 完成 INT8 matmul 后将 INT32 结果写入 workspace；Vector 从 workspace 读取 INT32，经过反量化后输出 BF16。

### 任务二：实现A8W8输入INT32输出的kernel

### 5.2 Path 1: QmmCubeBasicKernel

计算公式：
```
result_int32 = x1_int8 @ x2_int8
```

In [ ]:
%%writefile {QMM_CUSTOM_DIR}/op_kernel/qmm_custom.cpp

#include "kernel_operator.h"
#include "lib/matmul_intf.h"
#include "qmm_custom_tiling.h"

# TODO: 自行实现kernel

class QmmCubeBasicKernel {
public:
    __aicore__ inline QmmCubeBasicKernel() {}

    __aicore__ inline void Init(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR y,
                                GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe);

    __aicore__ inline void Process();
};

__aicore__ inline void QmmCubeBasicKernel::Init(
    GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR y,
    GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData, TPipe *tPipe)
{
    # TODO: 实现kernel初始化
}

__aicore__ inline void QmmCubeBasicKernel::Process()
{
    # TODO: 实现kernel执行
}

### 任务三：实现A8W8输入，经过perChannelScale和perTokenScale反量化输出BF16的kernel
### 5.3 Path 2: QmmPertokenKernel

计算公式：
```
result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale
```

In [ ]:
%%writefile -a {QMM_CUSTOM_DIR}/op_kernel/qmm_custom.cpp
# TODO: 自行实现kernel

class QmmPertokenKernel {
public:
    __aicore__ inline QmmPertokenKernel() {}

    __aicore__ inline void Init(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
                                GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData,
                                TPipe *tPipe);

    __aicore__ inline void Process();
};

__aicore__ inline void QmmPertokenKernel::Init(
    GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
    GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData,
    TPipe *tPipe)
{
    # TODO: 实现kernel初始化
}

__aicore__ inline void QmmPertokenKernel::Process()
{
    # TODO: 实现kernel执行
}

### 5.4 核函数入口

核函数根据 `isPertoken` 选择执行路径：

In [ ]:
%%writefile -a {QMM_CUSTOM_DIR}/op_kernel/qmm_custom.cpp
extern "C" __global__ __aicore__ void qmm_custom(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertoken_scale, GM_ADDR y, GM_ADDR workspace, GM_ADDR tiling) {
    if (workspace == nullptr) {
        return;
    }
    TPipe pipe;
    
    REGISTER_TILING_DEFAULT(QmmCustomTilingData);
    GET_TILING_DATA(tilingData, tiling);

    if (tilingData.isPertoken == 0) {
        QmmCubeBasicKernel kernel;
        kernel.Init(x1, x2, scale, y, workspace, &tilingData, &pipe);
        kernel.Process();
        pipe.Destroy();
    } else {
        QmmPertokenKernel kernel;
        kernel.Init(x1, x2, scale, pertoken_scale, y, workspace, &tilingData, &pipe);
        kernel.Process();
        pipe.Destroy();
    }
}

---
## 6 编译并安装自定义算子包

### 任务四：成功编译并安装自定义算子包

### 6.1 编译自定义算子包

在算子工程目录下执行编译脚本：

In [ ]:
!cd {QMM_CUSTOM_DIR} && bash build.sh

编译成功后可看到类似如下输出：

![image.png](images/compile_success.png)

### 6.2 安装自定义算子包

编译成功后，需要将打包生成的自定义算子包安装到环境中：

In [ ]:
!cd {QMM_CUSTOM_DIR}/build_out && chmod +x *.run && ./*.run

---

## 7 基于C++扩展调用自定义算子

算子编译部署完成后，还需要通过 PyTorch C++ Extension 将算子接入框架，才能在 Python 中调用。

该章节无需自行实现，仅为介绍性内容，直接使用提供的工程即可。

### 7.1 C++ Extension 工程结构

```
torch_ops_extension/
|--- build_and_install.sh               # 一键编译安装脚本
|--- setup.py                           # NpuExtension 构建配置
|--- custom_ops/                        # Python包 + C++源码
|   |--- __init__.py                    # 包初始化，自动挂载算子到 torch_npu
|   |--- converter/                     # python侧实现
|   |   |--- __init__.py
|   |   |--- qmm_custom.py              # QmmCustom 算子 GE converter
|   |--- csrc/                          # C++侧实现
|   |   |--- ops_common.h               # 公共工具库头文件
|   |   |--- ops_common.cpp             # 公共工具库实现
|   |   |--- ops_def_registration.cpp   # 算子定义与 pybind 注册
|   |   |--- qmm_custom.cpp             # 算子 NPU/META 实现
```

**关键文件说明**：
- `csrc/ops_def_registration.cpp`：算子定义 + pybind 注册
- `csrc/qmm_custom.cpp`：算子实现与设备注册
- `converter/qmm_custom.py`：torchair converter，用于 `torch.compile` 图模式
- `__init__.py`：导入编译生成的 C++ 扩展模块和 Python 接口模块；自动将 `torch.ops.custom.*` 挂载到 `torch_npu`，支持 `torch_npu.qmm_custom()` 调用方式

运行如下命令查看目录结构：

In [ ]:
!cd {QMM_EXT_DIR}; find . -maxdepth 3 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

### 7.2 构建和安装自定义算子的C++扩展

工程提供了 `build_and_install.sh` 一键编译安装脚本，该脚本执行以下步骤：
1. 清理历史编译产物
2. 设置 Ninja 编译和并行度
3. 编译 wheel 包（`setup.py build_ext` + `bdist_wheel`）
4. 安装 wheel 包到本地环境

In [ ]:
import subprocess, sys, shutil
import os

# 清理历史编译产物
build_dir = QMM_EXT_DIR / 'build'
if build_dir.exists(): shutil.rmtree(build_dir)

# 使用当前 kernel 的 Python（sys.executable）替代系统 python3
env = {**os.environ, 'USE_NINJA': '1', 'MAX_JOBS': '32'}

print(f'使用 Python: {sys.executable}')

# 编译 wheel
r1 = subprocess.run([sys.executable, 'setup.py', 'build_ext'], cwd=str(QMM_EXT_DIR), env=env)
if r1.returncode != 0: raise RuntimeError('build_ext 失败')
r2 = subprocess.run([sys.executable, 'setup.py', 'bdist_wheel'], cwd=str(QMM_EXT_DIR), env=env)
if r2.returncode != 0: raise RuntimeError('bdist_wheel 失败')

# 安装 wheel
dist_dir = QMM_EXT_DIR / 'dist'
whl_files = list(dist_dir.glob('*.whl'))
if not whl_files: raise RuntimeError('未找到 wheel 文件')
r3 = subprocess.run([sys.executable, '-m', 'pip', 'install', str(whl_files[0]), '-I'], env=env)
if r3.returncode != 0: raise RuntimeError('pip install 失败')
print('构建安装完成')

---
## 8 测试自定义算子功能性能

### 任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程

安装完成后，使用测试脚本验证 QmmCustom 算子在不同规格输入下的精度和性能。测试脚本覆盖多组 (M, K, N) 规格，分别验证 INT32 输出（无 perTokenScale）和 BF16 输出（有 perTokenScale）两种路径。

核心接口调用如下：

```python
output_npu = torch.ops.custom.qmm_custom(x1_npu, x2_npu, scale_npu, perTokenScale_npu, dtype)
```

其中 `dtype=1` 表示 BF16 输出，`dtype=0` 表示 INT32 输出。`perTokenScale_npu` 为 None 时走 INT32 路径。

### 8.1 测试算子功能

需要测试的 cases 的 shape 如下，**全部通过**视为算子功能OK：
| M    | K     | N     |
| ---- | ----- | ----- |
| 1    | 4096  | 4096  |
| 1    | 4096  | 6144  |
| 1    | 4096  | 24576 |
| 1    | 12288 | 4096  |
| 50   | 4096  | 4096  |
| 50   | 4096  | 6144  |
| 50   | 4096  | 24576 |
| 50   | 12288 | 4096  |
| 4096 | 4096  | 4096  |
| 4096 | 4096  | 6144  |
| 4096 | 4096  | 24576 |
| 4096 | 12288 | 4096  |

In [ ]:
import torch
import torch_npu
import custom_ops

torch.npu.config.allow_internal_format = True
torch.npu.config.allow_hf32 = False
torch.npu.set_compile_mode(jit_compile=False)

# ============================================================
# 测试配置: 多组 (M, K, N) 规格
# ============================================================
TEST_SHAPES = [
    (1, 4096, 4096),
    (1, 4096, 6144),
    (1, 4096, 24576),
    (1, 12288, 4096),
    (50, 4096, 4096),
    (50, 4096, 6144),
    (50, 4096, 24576),
    (50, 12288, 4096),
    (4096, 4096, 4096),
    (4096, 4096, 6144),
    (4096, 4096, 24576),
    (4096, 12288, 4096),
]


def check_close(output, ref, dtype):
    """统一的精度比对函数，根据输出 dtype 使用不同的 tolerance"""
    if dtype == torch.int32:
        # 整数运算应精确匹配，tolerance 为 0
        rtol, atol = 0, 0
    else:
        # BF16 允许浮点舍入误差
        rtol, atol = 0.01, 0.01

    output_f = output.to(torch.float32)
    ref_f = ref.to(torch.float32)
    passed = torch.allclose(output_f, ref_f, rtol=rtol, atol=atol)

    label = 'INT32' if dtype == torch.int32 else 'BF16'
    print(f"  {label} allclose验证: {'PASS' if passed else 'FAIL'}")

    if not passed:
        print(f"  output[0,0:5] = {output[0, 0:5].tolist()}")
        print(f"  ref[0,0:5]    = {ref[0, 0:5].tolist()}")

    return passed


def test_int32(M, K, N):
    """无 pertoken_scale, 纯INT8 matmul, INT32 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)
    scale_npu = scale_cpu.npu()

    output_npu = torch.ops.custom.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale=None, dtype=0)
    output_cpu = output_npu.cpu()

    # float64 matmul 走 BLAS 优化, 比 int32 朴素乘加快很多;
    # float64 精度足够精确表示 int32 范围内的整数结果
    ref_output = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)

    assert output_cpu.shape == ref_output.shape, f"shape不一致: {output_cpu.shape} vs {ref_output.shape}"
    assert output_cpu.dtype == torch.int32, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_output, torch.int32)


def test_bf16(M, K, N):
    """有 pertoken_scale, INT8 matmul + dequant, BF16 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)
    pertoken_scale_cpu = torch.randn([M], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)
    scale_npu = scale_cpu.npu()
    pertoken_scale_npu = pertoken_scale_cpu.npu()

    output_npu = torch.ops.custom.qmm_custom(x1_npu, x2_npu, scale_npu,
                                        pertoken_scale=pertoken_scale_npu, dtype=1)
    output_cpu = output_npu.cpu()

    int32_result = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)
    ref_bf16 = (int32_result.to(torch.float32) * scale_cpu.unsqueeze(0) * pertoken_scale_cpu.unsqueeze(1)).to(torch.bfloat16)

    assert output_cpu.shape == ref_bf16.shape, f"shape不一致: {output_cpu.shape} vs {ref_bf16.shape}"
    assert output_cpu.dtype == torch.bfloat16, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_bf16, torch.bfloat16)


# ============================================================
# 主循环: 对每组 (M, K, N) 执行 INT32 和 BF16 测试
# ============================================================
int32_results = []
bf16_results = []

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")
print("=" * 60)


### 8.2 测试算子性能
基于torch_npu.profiler来测试算子性能，执行下面的code cell测试单算子性能数据：

In [ ]:
import os
import csv
from pathlib import Path

# ============================================================
# Profiler 配置
# ============================================================
PROF_OUTPUT_DIR = str(TUTORIAL_DIR / "prof_output")  # 采集数据的存放路径
WARMUP_ITERS = 0      # warmup 步数, 不采集数据

def parse_kernel_details(csv_path):
    """解析 kernel_details.csv, 提取 QmmCustom 每次执行的 Duration
    """
    rows = []
    with open(csv_path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row.get("Name", "")
            if name != "QmmCustom":
                continue

            # 解析 Input Shapes: 格式如 "1,4096;4096,4096;4096;"
            raw_shapes = row.get("Input Shapes", "").strip('"')
            parts = [p.strip() for p in raw_shapes.split(";")]
            x1_shape = parts[0].split(",")  # (M, K)
            x2_shape = parts[1].split(",")  # (K, N)
            M = int(x1_shape[0])
            K = int(x1_shape[1])
            N = int(x2_shape[1])

            # 从 Output Data Types 判断 dtype
            out_dtype_raw = row.get("Output Data Types", "").strip('"')
            dtype_tag = "INT32" if "INT32" in out_dtype_raw else "BF16"

            # Duration(us)
            duration_us = float(row.get("Duration(us)", "0") or "0")

            rows.append({
                "M": M, "K": K, "N": N,
                "dtype": dtype_tag,
                "duration_us": duration_us,
            })
    return rows


def find_kernel_details_csv(prof_dir):
    prof_path = Path(prof_dir)
    if not prof_path.exists():
        return None
    # 找出所有包含 ASCEND_PROFILER_OUTPUT/kernel_details.csv 的子目录
    candidates = []
    for subdir in sorted(prof_path.iterdir(), key=lambda d: d.name, reverse=True):
        csv_path = subdir / "ASCEND_PROFILER_OUTPUT" / "kernel_details.csv"
        if csv_path.exists():
            candidates.append(csv_path)
            break  # sorted reverse=True, 第一个就是最新的
    return str(candidates[0]) if candidates else None

# ============================================================
# 主流程: 先 warmup, 再开启 profiler 采集, 最后汇总
# ============================================================
int32_results = []
bf16_results = []

# warmup: 不采集数据, 让 NPU 稳定
if WARMUP_ITERS>0:
    print("=" * 60)
    print(f"Warmup: 运行 {WARMUP_ITERS} 轮不采集数据")
    print("=" * 60)
    for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
        test_int32(M, K, N)
        test_bf16(M, K, N)

# 创建 profiler
os.makedirs(PROF_OUTPUT_DIR, exist_ok=True)
profiler = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.NPU,
        torch_npu.profiler.ProfilerActivity.CPU,
    ],
    experimental_config=torch_npu.profiler._ExperimentalConfig(
        profiler_level=torch_npu.profiler.ProfilerLevel.Level1,
        aic_metrics=torch_npu.profiler.AiCMetrics.PipeUtilization,
    ),
    on_trace_ready=torch_npu.profiler.tensorboard_trace_handler(PROF_OUTPUT_DIR),
)

# 开启 profiler 采集
profiler.start()

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))
    profiler.step()

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))
    profiler.step()

# 停止 profiler
profiler.stop()

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")

# ============================================================
# Profiler 数据汇总: 每种规格每种 dtype 的 Duration(us)
# ============================================================
print()
print("=" * 70)
print("Profiler 数据汇总 (QmmCustom Duration)")
print("=" * 70)

csv_path = find_kernel_details_csv(PROF_OUTPUT_DIR)
print(f"原始数据路径：{csv_path}")
if csv_path:
    kernel_rows = parse_kernel_details(csv_path)
    if kernel_rows:
        # 按 (M, K, N) 分组, 每组内有 INT32 和 BF16
        shape_map = {}
        for kr in kernel_rows:
            key = (kr["M"], kr["K"], kr["N"])
            shape_map.setdefault(key, {})[kr["dtype"]] = kr["duration_us"]

        print(f"{'规格 (M,K,N)':>24} | {'INT32 Duration(us)':>18} | {'BF16 Duration(us)':>18}")
        print("-" * 70)
        for M, K, N in TEST_SHAPES:
            key = (M, K, N)
            int32_dur = shape_map.get(key, {}).get("INT32", "-")
            bf16_dur = shape_map.get(key, {}).get("BF16", "-")
            int32_str = f"{int32_dur:.3f}" if isinstance(int32_dur, float) else int32_dur
            bf16_str = f"{bf16_dur:.3f}" if isinstance(bf16_dur, float) else bf16_dur
            tag = f"M={M},K={K},N={N}"
            print(f"{tag:>24} | {int32_str:>18} | {bf16_str:>18}")
    else:
        print(f"在 {csv_path} 中未找到 QmmCustom 算子数据")
else:
    print(f"未在 {PROF_OUTPUT_DIR} 中找到 kernel_details.csv 文件")
    print("请检查 profiler 输出目录")

print("=" * 70)


## 9 TIPS

上述内容基于notebook呈现了实现并测试自定义算子 QmmCustom 的详细步骤。整体过程如下图所示：
![process.png](images/process.png)

完成了一遍上述过程之后，可以按照如下步骤快速进行算子的修改与测试：

1. 修改算子实现源码文件，即op_host\qmm_custom.cpp、op_kernel\qmm_custom_tiling.h与op_kernel\qmm_custom.h
2. 编译并安装自定义算子包
3. 执行测试脚本测试功能与性能

提供了如下一键编译与测试脚本：

In [ ]:
!cat {SRC_DIR}/test.sh

## 10 评分标准总览

| 考核项 | 分值 | 评分说明 |
|------|------|--------|
| 功能正确性 | 10分 | (1)单算子测试集功能正确 (2)量化模型功能正确 |
| 性能 | 15分 | 量化模型性能PK，各组性能按排名赋分 |
| 代码提交 | 5分 | (1)代码规范性检查 (2)代码结构清晰 |
| 答辩 | 10分 | (1)方案设计 (2)优化过程 (3)后续优化方向 |

> 性能分须先通过算子功能验证（未通过则性能 0 分）。性能随机器/核数可能略有浮动，助教保留按实际重测结果调整的权利。

---

### 作业提交要求

1. 提交完整的 Jupyter Notebook（.ipynb），**保留所有单元格输出**，助教会重新运行评测。
2. 提交前依次运行所有单元格，确保：
   - 自定义算子编译成功；
   - 算子功能测试通过；
   - 算子性能数据收集成功。
3. **文件命名**：`gitcode账号_assignment_cann.ipynb`
4. **允许并且鼓励**使用 AI 工具辅助，但须独立思考并在报告中简述（解决了什么疑惑、是否引入新 bug）
5. **不得抄袭。**

---
### 小结

本章完成了 QmmCustom 算子的开发、编译部署、C++扩展接入和单算子功能性能测试。下一章会将该自定义算子接入 Qwen3-8B 量化模型，验证功能并观测其在模型中的性能。
